# Important warning!
Direct DICOM query of PACS can create high traffic or consume disproportionate PACS resources if you're not careful in setting up your query structure.
Adding things like time delays and ensuring we don't query for information we don't need are important ways to reduce the burden on the system.
Check in with Chris if you want to vet your script and look for ways to optimise or limit the resource usage.

Secondarily, note that PACS is a secure repository for sensitive patient data, so take care with how data removed from PACS is stored.

In [ ]:
from medphunc.pacs import thanks

import io

medphunc is my own library of DR-related scripts and functions. It's open source. I find it easier to use than basic pynetdicom. But the documentation isn't as good.

medphunc loads some libraries that store info about the PACS system. If you want to use medphunc on a different computer, you need to copy/create the yml file that specifies all of the different AE titles, defaults, orthanc location, etc.

Lets jump into it.

In [ ]:
#Check the configured remote:
thanks.pi.REMOTE
#This needs to be updated in aeinfo.yml, which is stored in the package directory. Swap between the configured options using:
#thanks.pi.REMOTE.interactive_set_ae_from_saved()
#The values on the Maryam webserver should work without adjustment.

# Case 1: Query a particular study
A common case that occurs might be that we want to get images/info from a particular study, where the details of that study are already known.
We always need to start our searches on the study level (if using AGFA PACS, other PACS vendors may allow multi-level queries. But lets not get too far into that right now.
The parameters (in the case below AccessionNumber) are based on the tags associated with the pydicom dataset.

In [ ]:
study_search = thanks.Thank(query_level='study', AcceStudyDate='20260422')
study_result = study_search.find()
study_result

The medphunc search objects (both SearchSet and Thanks) return a pandas dataframe from a query. This gives us study-level info about the procedure, which includes all the patient-level info, such as ID (MRN), name, date, modalities in the study.
Now, we often want to get information about which series are in the study! When searching on the series level, we MUST include the study instance unique identifer tag.

The 'drill_down' method lets us create a search on the next lower level based on the index in the previous results.

In [ ]:
series_search = study_search.drill_down(0)
series_results = series_search.find()
series_results

We can see there are some series in the PACS database for this system. None of the tags are very descriptive! Let's see if we can find a row with modality = SR - that indicates that this series probably contains the RDSR for the study. Lets query on series level and have a look!

In [ ]:
sr_results = series_search.results.loc[lambda x: x.Modality=='SR']

rdsr_search = series_search.drill_down(sr_results.index[0])
rdsr_results = rdsr_search.find()
rdsr_results

To move a single instance, can use the 'move' convenience function. When move is not given an index number, it moves on the level of the search object (which would be instance in this case). When moved with an index, it moves of the level of the serach object but adds the unique identifier of the supplied item.

FYI, if we inspect the SOPClassUID, and know what they all mean, we can now verify that this is in fact an RDSR object (SOP class 	1.2.840.10008.5.1.4.1.1.88.67).

In [ ]:
rdsr_search.move(0)

The data has been moved to orthanc! We can retrieve it via one of the orthanc commands. The retrieve method will return all the dicom objects in Orthanct which meets the search object criteria. The structure depends on the level. On the instance level, we get a list of pydicom objects.
On the series level we get a list of lists.
On the study level, a list of lists of lists.

In [ ]:
dcms = rdsr_search.retrieve(0)

In [ ]:
# grab the first (and only) dicom object. We expect a list one-deep:
d = dcms[0]

Now we can employ a convenience function for looking at the RDSR. They're a bit awkward to navigate in their native state.
This can be pointed at either a dicom object loaded into python, or a file path location. Again, the results will be a dictionary containing pandas dataframes.

In [ ]:
from medphunc.parsers import parse_rdsr
rdsr_content = parse_rdsr.dump_rdsr(d, 'temp.xlsx')

In [ ]:
rdsr_content.keys()
# dose data has the useful info about each scan. rdsr_metadata contains totals, and dicom metadata contains all the dicom tags.

In [ ]:
rdsr_content['dose_data']

We can now see the dose data for two localisers, as expected for this case

Now, lets look at some images. Lets get all the images from the first series.

In [ ]:
series_search.move(0)

In [ ]:
# Great - now retrieve all the results. Since we're moving on the series level, we expect a list of lists.
dcms = series_search.retrieve(0)
d = dcms[0][0]

In [ ]:
from matplotlib import pyplot as plt
from medphunc.image_analysis import image_utility as iu

In [ ]:
plt.imshow(d.pixel_array)

Almost impossible to see! That's because we haven't displayed using the window/level specified in the file, and we're showing the whole dynamic range of the image. We also haven't applied a more typical greyscale colormap.

In [ ]:
from medphunc.image_analysis import image_utility as iu
level = float(dcm_1.WindowCenter)
window = float(dcm_1.WindowWidth)
vmin, vmax = level-window, level+window
plt.imshow(dcm_1.pixel_array, vmin=vmin, vmax=vmax, cmap='bone')

In [ ]:
# Simple threshold based on intensity
test_im = dcm_1.pixel_array.copy() # Copy so we don't overwrite the original.
threshold = dcm_1.pixel_array.max()*0.25
mask = test_im > threshold
plt.imshow(mask)

Not bad, but we're getting a lot of torso content. Just dump everything in the top half of the image, with index < 700

In [ ]:
mask[0:700,:] = False

In [ ]:
plt.imshow(mask)

looks pretty good. We could tidy it up by binary filling holes and binary erosion of the smaller objects. But lets not bother, and just measure the area.

In [ ]:
mask.sum()*dcm_1.PixelSpacing[0]*dcm_1.PixelSpacing[1]

In mm^2. Not a useful value, but it's something to do.

By the way, there's a plotting convenience function in medphunc for dicom files in image_analysis.image_utility:

In [ ]:
iu.dcmshow(d)

# Case 2: we want information about lots of studies
The basic premise is the same, we just don't provide strictly limiting information in the original study query.

In [ ]:
multi_study_search = thanks.Thank('study',
                                  StudyDate='20240318-20240319', # date range, yearmonthday. You can set a single day, or a range.
                                  ModalitiesInStudy='CT', # must have the CT modality at minimum
                                  StudyDescription='*pine*' # * is wildcard, so include any study with 'pine' in the study description. With no wildcares, would be JUST studes described as pine, which would be none.
                                 )
multi_study_results = multi_study_search.find()
multi_study_results

15 spine CTs performed in 21-22 of September 2023. We used the wildcard search on study description

We can use the drill_all to get information about all the series for all the scans! This will require 17 series level searches behind the scenes, so beware the time and PACS burden of this approach can add up quickly.

In [ ]:
drill_results = multi_study_search.drill_all()

The resulting merged data can be accessed in the drill_merge property.

In [ ]:
multi_study_search.drill_merge

This will give you study+series level info in one dataframe, which can be searched using your normal pandas filtering, or inspected, or whatever suits really.

We can use the 'drill' function as a shortcut for going from study to series to instance level searches. drill_down only works AFTER a find has been performed.

In [ ]:
series_search_2 = multi_study_search.drill_down(3) # arbitrarily pick the 3rd result in multi_study_results. Automatically sets studyinstanceuid
series_search_2.find()

There's also some other convenience functions, like 'move_one_instance_all_series', which, when applied to a study level search does exactly that. drills down into each series, and moves exactly one dicom file from each, which can be useful for getting lots of dicom metadata tags. Again, this can lead to a LOT of server usage depending on how wide the original search object is, so please use with care!